In [1]:
import numpy as np
from typing import List, Union, Tuple, Optional
import warnings
from numba import jit, cuda, complex128, int32, float64
from numba.cuda import random
import math


In [2]:

# Try to import CuPy for GPU arrays (fallback to NumPy if not available)
try:
    import cupy as cp
    GPU_AVAILABLE = cuda.is_available()
    print(f"GPU Support: {'Available' if GPU_AVAILABLE else 'Not Available'}")
    if GPU_AVAILABLE:
        print(f"GPU Device: {cuda.get_current_device().name}")
except ImportError:
    cp = None
    GPU_AVAILABLE = False
    print("CuPy not available. GPU features disabled.")


CuPy not available. GPU features disabled.


In [3]:

# Numba-compiled helper functions for performance-critical operations
@jit(nopython=True, cache=True)
def apply_single_qubit_gate_numba(statevector, gate_matrix, target_qubit, num_qubits):
    """Apply single-qubit gate using Numba JIT compilation."""
    num_states = len(statevector)
    new_statevector = np.zeros_like(statevector)
    
    for state_idx in range(num_states):
        # Extract target qubit value
        target_bit = (state_idx >> target_qubit) & 1
        
        # Calculate the complementary state (flip target bit)
        complement_idx = state_idx ^ (1 << target_qubit)
        
        if target_bit == 0:
            # |0⟩ component
            new_statevector[state_idx] += gate_matrix[0, 0] * statevector[state_idx]
            new_statevector[state_idx] += gate_matrix[0, 1] * statevector[complement_idx]
        else:
            # |1⟩ component  
            new_statevector[state_idx] += gate_matrix[1, 0] * statevector[complement_idx]
            new_statevector[state_idx] += gate_matrix[1, 1] * statevector[state_idx]
    
    return new_statevector

@jit(nopython=True, cache=True)
def apply_cnot_gate_numba(statevector, control_qubit, target_qubit):
    """Apply CNOT gate using optimized Numba implementation."""
    num_states = len(statevector)
    new_statevector = statevector.copy()
    
    for state_idx in range(num_states):
        # Check if control qubit is |1⟩
        control_bit = (state_idx >> control_qubit) & 1
        
        if control_bit == 1:
            # Flip target qubit
            flipped_idx = state_idx ^ (1 << target_qubit)
            # Swap amplitudes
            temp = new_statevector[state_idx]
            new_statevector[state_idx] = new_statevector[flipped_idx]
            new_statevector[flipped_idx] = temp
    
    return new_statevector

@jit(nopython=True, cache=True)
def calculate_probabilities_numba(statevector):
    """Calculate measurement probabilities with Numba optimization."""
    return np.abs(statevector) ** 2

@jit(nopython=True, cache=True)
def calculate_expectation_pauli_z_numba(statevector, qubit_idx, num_qubits):
    """Calculate expectation value of Pauli-Z operator."""
    expectation = 0.0 + 0.0j
    num_states = len(statevector)
    
    for state_idx in range(num_states):
        bit_value = (state_idx >> qubit_idx) & 1
        eigenvalue = 1.0 if bit_value == 0 else -1.0
        expectation += np.conj(statevector[state_idx]) * eigenvalue * statevector[state_idx]
    
    return expectation.real

@jit(nopython=True, cache=True)
def normalize_statevector_numba(statevector):
    """Normalize statevector with Numba optimization."""
    norm = 0.0
    for i in range(len(statevector)):
        norm += np.abs(statevector[i]) ** 2
    norm = np.sqrt(norm)
    
    if norm > 1e-15:
        for i in range(len(statevector)):
            statevector[i] /= norm
    
    return statevector

# GPU kernel for single-qubit gate application
if GPU_AVAILABLE:
    @cuda.jit
    def apply_single_qubit_gate_gpu(statevector, new_statevector, gate_real, gate_imag, target_qubit):
        """GPU kernel for applying single-qubit gates."""
        idx = cuda.grid(1)
        if idx >= statevector.shape[0]:
            return
        
        # Extract target qubit value
        target_bit = (idx >> target_qubit) & 1
        complement_idx = idx ^ (1 << target_qubit)
        
        if target_bit == 0:
            # |0⟩ component
            real_part = (gate_real[0, 0] * statevector[idx].real - gate_imag[0, 0] * statevector[idx].imag +
                        gate_real[0, 1] * statevector[complement_idx].real - gate_imag[0, 1] * statevector[complement_idx].imag)
            imag_part = (gate_real[0, 0] * statevector[idx].imag + gate_imag[0, 0] * statevector[idx].real +
                        gate_real[0, 1] * statevector[complement_idx].imag + gate_imag[0, 1] * statevector[complement_idx].real)
            new_statevector[idx] = complex(real_part, imag_part)
        else:
            # |1⟩ component
            real_part = (gate_real[1, 0] * statevector[complement_idx].real - gate_imag[1, 0] * statevector[complement_idx].imag +
                        gate_real[1, 1] * statevector[idx].real - gate_imag[1, 1] * statevector[idx].imag)
            imag_part = (gate_real[1, 0] * statevector[complement_idx].imag + gate_imag[1, 0] * statevector[complement_idx].real +
                        gate_real[1, 1] * statevector[idx].imag + gate_imag[1, 1] * statevector[idx].real)
            new_statevector[idx] = complex(real_part, imag_part)


In [4]:

class QuantumSimulator:
    """
    High-performance quantum statevector simulator with Numba JIT compilation and GPU support.
    """
    
    def __init__(self, num_qubits: int, use_gpu: bool = False):
        """
        Initialize quantum simulator.
        
        Args:
            num_qubits: Number of qubits in the quantum system
            use_gpu: Whether to use GPU acceleration (requires CuPy and CUDA)
        """
        if num_qubits <= 0:
            raise ValueError("Number of qubits must be positive")
        if num_qubits > 25:
            warnings.warn("Large number of qubits may cause memory issues")
            
        self.num_qubits = num_qubits
        self.num_states = 2 ** num_qubits
        self.use_gpu = use_gpu and GPU_AVAILABLE
        
        if self.use_gpu and not GPU_AVAILABLE:
            warnings.warn("GPU requested but not available. Falling back to CPU.")
            self.use_gpu = False
        
        # Choose array library based on GPU availability
        self.xp = cp if self.use_gpu else np
        
        # Initialize to |0...0⟩ state
        self.statevector = self.xp.zeros(self.num_states, dtype=self.xp.complex128)
        self.statevector[0] = 1.0
        
        # Pre-compute common gates
        self._precompute_gates()
        
        # Cache for rotation gates
        self._rotation_cache = {}
        
        print(f"Quantum Simulator initialized: {num_qubits} qubits, "
              f"{'GPU' if self.use_gpu else 'CPU'} backend")
    
    def _precompute_gates(self):
        """Pre-compute common quantum gates."""
        # Use appropriate array library
        xp = self.xp
        
        # Pauli gates
        self.I = xp.array([[1, 0], [0, 1]], dtype=xp.complex128)
        self.X = xp.array([[0, 1], [1, 0]], dtype=xp.complex128)
        self.Y = xp.array([[0, -1j], [1j, 0]], dtype=xp.complex128)
        self.Z = xp.array([[1, 0], [0, -1]], dtype=xp.complex128)
        
        # Hadamard gate
        self.H = xp.array([[1, 1], [1, -1]], dtype=xp.complex128) / xp.sqrt(2)
        
        # Phase gates
        self.S = xp.array([[1, 0], [0, 1j]], dtype=xp.complex128)
        self.T = xp.array([[1, 0], [0, xp.exp(1j * xp.pi / 4)]], dtype=xp.complex128)
    
    def to_cpu(self):
        """Transfer computation to CPU."""
        if self.use_gpu:
            self.statevector = cp.asnumpy(self.statevector)
            self.use_gpu = False
            self.xp = np
            self._precompute_gates()
            print("Switched to CPU backend")
    
    def to_gpu(self):
        """Transfer computation to GPU."""
        if GPU_AVAILABLE and not self.use_gpu:
            self.statevector = cp.asarray(self.statevector)
            self.use_gpu = True
            self.xp = cp
            self._precompute_gates()
            print("Switched to GPU backend")
        elif not GPU_AVAILABLE:
            print("GPU not available")
    
    def reset(self):
        """Reset the quantum system to |0...0⟩ state."""
        self.statevector = self.xp.zeros(self.num_states, dtype=self.xp.complex128)
        self.statevector[0] = 1.0
    
    def get_statevector(self) -> np.ndarray:
        """Return a copy of the current statevector as NumPy array."""
        if self.use_gpu:
            return cp.asnumpy(self.statevector.copy())
        return self.statevector.copy()
    
    def get_probabilities(self) -> np.ndarray:
        """Get measurement probabilities for all basis states."""
        if self.use_gpu:
            probs = cp.abs(self.statevector) ** 2
            return cp.asnumpy(probs)
        else:
            # Use Numba-optimized version for CPU
            statevector_cpu = np.asarray(self.statevector)
            return calculate_probabilities_numba(statevector_cpu)
    
    def _rotation_gate(self, gate_type: str, angle: float):
        """Create rotation gate matrices with caching."""
        cache_key = (gate_type, angle, self.use_gpu)
        
        if cache_key in self._rotation_cache:
            return self._rotation_cache[cache_key]
        
        xp = self.xp
        cos_half = xp.cos(angle / 2)
        sin_half = xp.sin(angle / 2)
        
        if gate_type == 'RX':
            gate = xp.array([[cos_half, -1j * sin_half],
                           [-1j * sin_half, cos_half]], dtype=xp.complex128)
        elif gate_type == 'RY':
            gate = xp.array([[cos_half, -sin_half],
                           [sin_half, cos_half]], dtype=xp.complex128)
        elif gate_type == 'RZ':
            gate = xp.array([[xp.exp(-1j * angle / 2), 0],
                           [0, xp.exp(1j * angle / 2)]], dtype=xp.complex128)
        else:
            raise ValueError(f"Unknown rotation gate: {gate_type}")
        
        self._rotation_cache[cache_key] = gate
        return gate
    
    def apply_gate(self, gate_name: str, target_qubit: int, **kwargs):
        """Apply a quantum gate with optimized implementations."""
        gate_name = gate_name.upper()
        
        if target_qubit >= self.num_qubits or target_qubit < 0:
            raise ValueError(f"Invalid target qubit: {target_qubit}")
        
        if gate_name == 'CNOT':
            control_qubit = kwargs.get('control')
            if control_qubit is None:
                raise ValueError("CNOT gate requires control qubit")
            self._apply_cnot(control_qubit, target_qubit)
            
        elif gate_name in ['X', 'Y', 'Z', 'H', 'S', 'T']:
            gate_matrix = getattr(self, gate_name)
            self._apply_single_qubit_gate(gate_matrix, target_qubit)
            
        elif gate_name in ['RX', 'RY', 'RZ']:
            angle = kwargs.get('angle', 0)
            gate_matrix = self._rotation_gate(gate_name, angle)
            self._apply_single_qubit_gate(gate_matrix, target_qubit)
            
        else:
            raise ValueError(f"Unsupported gate: {gate_name}")
    
    def _apply_single_qubit_gate(self, gate_matrix, target_qubit: int):
        """Apply single-qubit gate with backend-specific optimization."""
        if self.use_gpu and GPU_AVAILABLE:
            self._apply_single_qubit_gate_gpu(gate_matrix, target_qubit)
        else:
            self._apply_single_qubit_gate_cpu(gate_matrix, target_qubit)
    
    def _apply_single_qubit_gate_cpu(self, gate_matrix, target_qubit: int):
        """CPU-optimized single-qubit gate application using Numba."""
        # Convert to numpy for Numba processing
        statevector_np = np.asarray(self.statevector)
        gate_np = np.asarray(gate_matrix)
        
        # Apply gate using Numba-optimized function
        new_statevector = apply_single_qubit_gate_numba(
            statevector_np, gate_np, target_qubit, self.num_qubits
        )
        
        # Normalize and update
        self.statevector = normalize_statevector_numba(new_statevector)
    
    def _apply_single_qubit_gate_gpu(self, gate_matrix, target_qubit: int):
        """GPU-optimized single-qubit gate application."""
        if not self.use_gpu:
            return self._apply_single_qubit_gate_cpu(gate_matrix, target_qubit)
        
        # Prepare GPU arrays
        new_statevector = cp.zeros_like(self.statevector)
        gate_real = cp.real(gate_matrix)
        gate_imag = cp.imag(gate_matrix)
        
        # Calculate grid dimensions
        threads_per_block = 256
        blocks_per_grid = (self.num_states + threads_per_block - 1) // threads_per_block
        
        # Launch GPU kernel
        apply_single_qubit_gate_gpu[blocks_per_grid, threads_per_block](
            self.statevector, new_statevector, gate_real, gate_imag, target_qubit
        )
        
        # Normalize and update
        self.statevector = new_statevector
        norm = cp.linalg.norm(self.statevector)
        if norm > 1e-15:
            self.statevector /= norm
    
    def _apply_cnot(self, control_qubit: int, target_qubit: int):
        """Apply CNOT gate with optimized implementation."""
        if control_qubit == target_qubit:
            raise ValueError("Control and target qubits must be different")
        if (control_qubit >= self.num_qubits or target_qubit >= self.num_qubits or 
            control_qubit < 0 or target_qubit < 0):
            raise ValueError("Invalid qubit indices")
        
        if self.use_gpu:
            # GPU implementation would be more complex, using CPU for now
            statevector_cpu = cp.asnumpy(self.statevector)
            new_statevector = apply_cnot_gate_numba(statevector_cpu, control_qubit, target_qubit)
            self.statevector = cp.asarray(new_statevector)
        else:
            # CPU Numba-optimized implementation
            statevector_np = np.asarray(self.statevector)
            self.statevector = apply_cnot_gate_numba(statevector_np, control_qubit, target_qubit)
    
    def measure_all(self, shots: int = 1) -> List[str]:
        """Perform measurement on all qubits."""
        probabilities = self.get_probabilities()
        
        outcomes = []
        for _ in range(shots):
            outcome_idx = np.random.choice(self.num_states, p=probabilities)
            binary_outcome = format(outcome_idx, f'0{self.num_qubits}b')
            outcomes.append(binary_outcome)
        
        return outcomes
    
    def get_expectation_value(self, observable: str, qubit_idx: int) -> float:
        """Calculate expectation value of Pauli observable with optimization."""
        observable = observable.upper()
        
        if observable == 'Z':
            if self.use_gpu:
                # Transfer to CPU for Numba processing
                statevector_cpu = cp.asnumpy(self.statevector)
                return calculate_expectation_pauli_z_numba(statevector_cpu, qubit_idx, self.num_qubits)
            else:
                statevector_np = np.asarray(self.statevector)
                return calculate_expectation_pauli_z_numba(statevector_np, qubit_idx, self.num_qubits)
        
        elif observable in ['X', 'Y']:
            # For X and Y, we need to apply the gate first (less optimized for now)
            gate_matrix = getattr(self, observable)
            
            if self.use_gpu:
                # Create full operator (expensive, consider optimization)
                operator = self._create_single_qubit_operator_gpu(gate_matrix, qubit_idx)
                expectation = cp.conj(self.statevector) @ operator @ self.statevector
                return float(cp.real(expectation))
            else:
                operator = self._create_single_qubit_operator_cpu(gate_matrix, qubit_idx)
                expectation = np.conj(self.statevector) @ operator @ self.statevector
                return float(np.real(expectation))
        
        else:
            raise ValueError("Observable must be 'X', 'Y', or 'Z'")
    
    def _create_single_qubit_operator_cpu(self, gate: np.ndarray, target_qubit: int) -> np.ndarray:
        """Create full system operator for CPU."""
        operators = []
        I_np = np.array([[1, 0], [0, 1]], dtype=np.complex128)
        
        for i in range(self.num_qubits):
            if i == target_qubit:
                operators.append(gate)
            else:
                operators.append(I_np)
        
        result = operators[0]
        for op in operators[1:]:
            result = np.kron(result, op)
        
        return result
    
    def _create_single_qubit_operator_gpu(self, gate, target_qubit: int):
        """Create full system operator for GPU."""
        operators = []
        I_gpu = cp.array([[1, 0], [0, 1]], dtype=cp.complex128)
        
        for i in range(self.num_qubits):
            if i == target_qubit:
                operators.append(gate)
            else:
                operators.append(I_gpu)
        
        result = operators[0]
        for op in operators[1:]:
            result = cp.kron(result, op)
        
        return result
    
    def benchmark(self, num_operations: int = 1000):
        """Benchmark the simulator performance."""
        import time
        
        print(f"\nBenchmarking {num_operations} random gate operations...")
        print(f"Backend: {'GPU' if self.use_gpu else 'CPU'}")
        
        gates = ['X', 'Y', 'Z', 'H', 'S', 'T']
        
        start_time = time.time()
        
        for _ in range(num_operations):
            gate = np.random.choice(gates)
            qubit = np.random.randint(0, self.num_qubits)
            self.apply_gate(gate, qubit)
        
        end_time = time.time()
        
        total_time = end_time - start_time
        ops_per_second = num_operations / total_time
        
        print(f"Total time: {total_time:.4f} seconds")
        print(f"Operations per second: {ops_per_second:.2f}")
        print(f"Average time per operation: {total_time/num_operations*1000:.4f} ms")
    
    def __str__(self) -> str:
        """String representation of the quantum state."""
        statevector_cpu = self.get_statevector()  # Always get CPU version for display
        
        result = f"Quantum State ({self.num_qubits} qubits, {'GPU' if self.use_gpu else 'CPU'}):\n"
        
        # Show only significant amplitudes
        for i, amplitude in enumerate(statevector_cpu):
            if np.abs(amplitude) > 1e-10:
                binary_state = format(i, f'0{self.num_qubits}b')
                prob = np.abs(amplitude) ** 2
                result += f"|{binary_state}⟩: {amplitude:.4f} (prob: {prob:.4f})\n"
        
        return result



In [5]:

# Example usage and benchmarking
print("=== Quantum Simulator Performance Test ===")

# Test with different backends
num_qubits = 12  # Start with moderate size

print(f"\n--- CPU Backend ---")
sim_cpu = QuantumSimulator(num_qubits, use_gpu=False)

# Create entangled state
sim_cpu.apply_gate('H', 0)
for i in range(1, min(4, num_qubits)):  # Create some entanglement
    sim_cpu.apply_gate('CNOT', target_qubit=i, control=i-1)

print("Created entangled state:")
if num_qubits <= 4:  # Only print for small systems
    print(sim_cpu)

# Benchmark CPU
sim_cpu.benchmark(100)

if GPU_AVAILABLE:
    print(f"\n--- GPU Backend ---")
    sim_gpu = QuantumSimulator(num_qubits, use_gpu=True)
    
    # Create same entangled state
    sim_gpu.apply_gate('H', 0)
    for i in range(1, min(4, num_qubits)):
        sim_gpu.apply_gate('CNOT', target_qubit=i, control=i-1)
    
    # Benchmark GPU
    sim_gpu.benchmark(100)
    
    print(f"\n--- Switching Backends ---")
    print("Moving from GPU to CPU...")
    sim_gpu.to_cpu()
    print("Moving from CPU to GPU...")
    sim_gpu.to_gpu()

print(f"\n--- Measurement Test ---")
sim_test = QuantumSimulator(3, use_gpu=GPU_AVAILABLE)
sim_test.apply_gate('H', 0)
sim_test.apply_gate('CNOT', target_qubit=1, control=0)

print("Bell state created:")
print(sim_test)

print("Measurement outcomes (10 shots):")
outcomes = sim_test.measure_all(10)
for i, outcome in enumerate(outcomes):
    print(f"Shot {i+1}: |{outcome}⟩")

print(f"\nExpectation values:")
print(f"⟨Z₀⟩ = {sim_test.get_expectation_value('Z', 0):.4f}")
print(f"⟨Z₁⟩ = {sim_test.get_expectation_value('Z', 1):.4f}")

=== Quantum Simulator Performance Test ===

--- CPU Backend ---
Quantum Simulator initialized: 12 qubits, CPU backend
Created entangled state:

Benchmarking 100 random gate operations...
Backend: CPU
Total time: 0.0023 seconds
Operations per second: 43049.41
Average time per operation: 0.0232 ms

--- Measurement Test ---
Quantum Simulator initialized: 3 qubits, CPU backend
Bell state created:
Quantum State (3 qubits, CPU):
|000⟩: 0.7071+0.0000j (prob: 0.5000)
|001⟩: 0.7071+0.0000j (prob: 0.5000)

Measurement outcomes (10 shots):
Shot 1: |000⟩
Shot 2: |000⟩
Shot 3: |001⟩
Shot 4: |001⟩
Shot 5: |000⟩
Shot 6: |001⟩
Shot 7: |000⟩
Shot 8: |001⟩
Shot 9: |001⟩
Shot 10: |000⟩

Expectation values:
⟨Z₀⟩ = 0.0000
⟨Z₁⟩ = 1.0000
